In [0]:
# Environment selection as dropdown
dbutils.widgets.dropdown(
    name="environment",
    defaultValue="fq_dev_pnl",
    choices=["fq_dev_pnl", "fq_test_pnl", "fq_prod_pnl"],
    label="Select environment"
)

# Source selection as combobox
dbutils.widgets.combobox(
    name="source",
    defaultValue="NETSUITE",
    choices=["POSIST", "NETSUITE", "other"],
    label="Source"
)

# Domain selection as combobox
dbutils.widgets.combobox(
    name="domain",
    defaultValue="management_pnl",
    choices=["management_pnl"],
    label="Domain"
)

environment = dbutils.widgets.get("environment")
source = dbutils.widgets.get("source")
domain = dbutils.widgets.get("domain")

staging = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `fq_dev_extloc_staging`"
).select("url").collect()[0][0]

checkpoint = 'abfss://fq-dev-pnl-container@fqadfstoragedev.dfs.core.windows.net/checkpoints/'

In [0]:
%skip
select * from fq_dev_pnl_catalog.bronze.gl_report limit 1

In [0]:
%run "/Workspace/Users/tgh3@foodquest.ae/FoodQuest_PnL.git/FoodQuest Management P&L/Formula & Functions Management P&L"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS fq_dev_pnl_catalog.silver.management_pnl (
  city STRING,
  management_sort_order INT,
  location_id INT,
  store_type STRING,
  major_group STRING,
  `Detail/Total` STRING COMMENT 'Detail/Total indicator',
  account_name STRING,
  zone STRING,
  sub_group STRING,
  management_details_total STRING,
  type STRING COMMENT 'Store/HO type',
  account_type STRING COMMENT 'Income/Expense type',
  store_open_date2 STRING,
  brand_id STRING,
  company_id STRING,
  parent_company STRING,
  country_code STRING,
  group STRING COMMENT 'Sales and Services Income group',
  management_group STRING,
  year INT,
  month STRING,
  netsuite_location_name STRING,
  mapped_name STRING,
  amount DECIMAL(18,2),
  budget_amount DECIMAL(18,2),
  py_amount DECIMAL(18,2) COMMENT 'Previous year amount'
)
USING DELTA
CLUSTER BY auto
LOCATION 'abfss://fq-dev-pnl-container@fqadfstoragedev.dfs.core.windows.net/silver/management_pnl' 
TBLPROPERTIES (
  'delta.columnMapping.mode' = 'name',
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
);

In [0]:
from delta.tables import DeltaTable
import time

def merge_stream_management_pnl(df, i):
    try:
        # Enrich data to include all necessary fields
        enriched = df_final_actual(df)
        management_pnl_upsert = df_final_all(enriched)
        management_pnl_upsert = management_pnl_upsert.dropDuplicates(["year", "month", "netsuite_location_name", "mapped_name"])

        target_table = DeltaTable.forName(df.sparkSession, "fq_dev_pnl_catalog.silver.management_pnl")

        merge_condition = """
            target.year = source.year
            AND target.month = source.month
            AND target.netsuite_location_name = source.netsuite_location_name
            AND target.mapped_name = source.mapped_name
        """

        (target_table.alias("target")
            .merge(management_pnl_upsert.alias("source"), merge_condition)
            .whenMatchedUpdate(set={
                "amount": "source.amount",
                "budget_amount": "source.budget_amount",
                "py_amount": "source.py_amount"
            })
            .whenNotMatchedInsertAll()
            .execute()
        )

        print(f"Successfully merged batch {i}")
        # print(f"Batch {i}: {df.count()} rows")

    except Exception as e:
        print(f"Error in merge_stream: {e}")
        raise e

# Streaming read and write
(spark.readStream
    .table("fq_dev_pnl_catalog.bronze.gl_report")
    .writeStream
    .foreachBatch(merge_stream_management_pnl)
    .option("mergeSchema", "true")
    .option("checkpointLocation", f'{checkpoint}/{source}/{domain}/streaming/checkpoint_silver_management_pnl')
    .trigger(availableNow=True)
    .start()
).awaitTermination()

In [0]:
%sql
SELECT 
  COUNT(*) AS total_rows,
  count(distinct month) as total_months,
  count(distinct year) as total_years,
  count(distinct netsuite_location_name) as total_locations
FROM fq_dev_pnl_catalog.silver.management_pnl;

In [0]:
%skip
select * from fq_dev_pnl_catalog.silver.management_pnl limit 1

In [0]:
for query in spark.streams.active:
    query.stop()

In [0]:
flag = True

if flag:
    dbutils.notebook.exit('Skipping up next demo cells')

In [0]:
df = spark.read.option('multiline', False).format('json').load('abfss://staging@fqadfstoragedev.dfs.core.windows.net/FoodQuest/Netsuite/GL_Report/DENNYS/*/*/gl_report.json')

df_final = df_final_actual(df)
management_pnl_upsert = df_final_all(df_final)



In [0]:
management_pnl_upsert.display()

In [0]:
# First, create a temporary view from your source DataFrame
management_pnl_upsert.createOrReplaceTempView("management_pnl_upsert_temp")

# Now execute the MERGE using SQL
merge_sql = """
MERGE INTO fq_dev_pnl_catalog.silver.management_pnl AS target
USING management_pnl_upsert_temp AS source
ON target.year = source.year
    AND target.month = source.month
    AND target.netsuite_location_name = source.netsuite_location_name
    AND target.mapped_name = source.mapped_name
WHEN MATCHED THEN
    UPDATE SET
        target.amount = source.amount,
        target.budget_amount = source.budget_amount,
        target.py_amount = source.py_amount
WHEN NOT MATCHED THEN
    INSERT *
"""

spark.sql(merge_sql)

print("✅ MERGE operation completed successfully")

In [0]:
%sql
SELECT *
FROM fq_dev_pnl_catalog.silver.management_pnl where netsuite_location_name is not null
ORDER BY parent_company, company_id, brand_id,
         netsuite_location_name, year, month, management_sort_order;